# CDS Project Part 3 — Vulnerability Prediction with BiLSTM/BiGRU Ensemble
*Institute of Software Security (E22) | Hamburg University of Technology | SoSe 2026*

## Task 1 — Model Architecture

We use a **3-model ensemble** of two BiLSTMs and one BiGRU, trained on raw token sequences from Java source code.

**Why BiLSTM/BiGRU?** Bidirectional recurrent networks capture long-range dependencies in both directions — a variable declared early in a function may be used unsafely much later. LSTM and GRU use different gating mechanisms, so ensembling them gives better coverage of different vulnerability patterns than any single model.

**Why ensemble?** Each model makes different errors. Averaging their predicted probabilities and tuning a classification threshold on the validation set yields higher F1 than any individual model.

**Preprocessing:** A regex-based tokenizer splits Java code into identifiers, numbers, and punctuation. A vocabulary of 20,000 tokens is built from the training set only (to avoid data leakage). Sequences are truncated to 400 tokens and padded within each batch.

## Setup

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

import os
DRIVE_DIR  = "/content/drive/MyDrive/CDS_Project"
TRAIN_PATH = os.path.join(DRIVE_DIR, "vuln_dataset.jsonl")
assert os.path.exists(TRAIN_PATH), f"Dataset not found at {TRAIN_PATH}"

!pip install scikit-learn matplotlib seaborn tqdm --quiet

import torch, json, re, copy, random, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn
from collections import Counter, defaultdict
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix, classification_report)
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
CHALLENGE_PATH = "/content/cds_challenge.jsonl"
if not os.path.exists(CHALLENGE_PATH):
    uploaded = files.upload()
    for name in uploaded:
        if name == "cds_challenge.jsonl":
            with open(CHALLENGE_PATH, "wb") as f:
                f.write(uploaded[name])

## Task 2 — Load and Split Dataset

We use a **70/15/15 stratified split** (train/validation/test). Three sets are needed because the validation set is used for early stopping and threshold tuning, while the test set provides an unbiased final evaluation. Stratification preserves the class ratio across all splits.

In [ ]:
functions, labels = [], []
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        functions.append(row["function"])
        labels.append(int(row["vulnerable"]))

challenge_ids, challenge_funcs = [], []
with open(CHALLENGE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        challenge_ids.append(row["vul_id"])
        challenge_funcs.append(row["func"])

X_temp, X_test, y_temp, y_test = train_test_split(
    functions, labels, test_size=0.15, stratify=labels, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)} | Challenge: {len(challenge_funcs)}")
print(f"Vulnerable: train={sum(y_train)}, val={sum(y_val)}, test={sum(y_test)}")

## Task 3 — Preprocessing, Training, and Evaluation

In [ ]:
TOKEN_RE   = re.compile(r"[A-Za-z_][A-Za-z_0-9]*|[0-9]+|[^\s]")
MAX_LEN    = 400
VOCAB_SIZE = 20000
BATCH_SIZE = 32

def tokenize(code):
    return TOKEN_RE.findall(code)[:MAX_LEN]

counter = Counter()
for code in X_train:
    counter.update(tokenize(code))

vocab = {"<pad>": 0, "<unk>": 1}
for tok, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[tok] = len(vocab)

PAD_IDX, UNK_IDX = 0, 1

def encode(code):
    return [vocab.get(t, UNK_IDX) for t in tokenize(code)]

print(f"Vocabulary size: {len(vocab)}")

In [ ]:
class VulnDataset(Dataset):
    def __init__(self, codes, labels):
        self.codes, self.labels = codes, labels
    def __len__(self):
        return len(self.codes)
    def __getitem__(self, idx):
        ids = encode(self.codes[idx]) or [UNK_IDX]
        return torch.tensor(ids, dtype=torch.long), float(self.labels[idx])

class ChallengeDataset(Dataset):
    def __init__(self, codes):
        self.codes = codes
    def __len__(self):
        return len(self.codes)
    def __getitem__(self, idx):
        ids = encode(self.codes[idx]) or [UNK_IDX]
        return torch.tensor(ids, dtype=torch.long), 0.0

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    padded = torch.full((len(seqs), int(lengths.max())), PAD_IDX, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = s
    return padded, lengths, torch.tensor(labels, dtype=torch.float)

train_loader = DataLoader(VulnDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(VulnDataset(X_val, y_val),     batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(VulnDataset(X_test, y_test),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, bidirectional=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim * 2, 1))

    def forward(self, x, lengths):
        emb = self.drop(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        mask = (x != PAD_IDX).unsqueeze(-1)[:, :out.size(1), :]
        out = out.masked_fill(~mask, float("-inf"))
        return self.head(out.max(dim=1)[0]).squeeze(-1)


class BiGRUClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=160, hidden_dim=160, num_layers=2, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim * 2, 1))

    def forward(self, x, lengths):
        emb = self.drop(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        mask = (x != PAD_IDX).unsqueeze(-1)[:, :out.size(1), :]
        out = out.masked_fill(~mask, float("-inf"))
        return self.head(out.max(dim=1)[0]).squeeze(-1)

In [ ]:
@torch.no_grad()
def get_probs(model, loader, device):
    model.eval()
    all_p = []
    for x, lengths, _ in loader:
        x = x.to(device)
        all_p.append(torch.sigmoid(model(x, lengths)).cpu().numpy())
    return np.concatenate(all_p)


def train_model(model, device, pos_weight_boost=2.0, epochs=15, patience=3):
    n_pos = sum(y_train)
    n_neg = len(y_train) - n_pos
    pw = torch.tensor([n_neg / max(n_pos, 1) * pos_weight_boost], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_precision": [], "val_recall": []}
    best_f1, best_state, no_improve = -1.0, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for x, lengths, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x, lengths), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * x.size(0)
        train_loss = total_loss / len(train_loader.dataset)

        # Validation
        val_probs = get_probs(model, val_loader, device)
        val_preds = (val_probs > 0.5).astype(int)
        val_ys = np.array(y_val)
        v_loss = nn.BCEWithLogitsLoss()(torch.tensor(np.log(val_probs / (1 - val_probs + 1e-8))), torch.tensor(val_ys, dtype=torch.float)).item()
        v_f1 = f1_score(val_ys, val_preds, zero_division=0)
        v_p  = precision_score(val_ys, val_preds, zero_division=0)
        v_r  = recall_score(val_ys, val_preds, zero_division=0)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(v_loss)
        history["val_f1"].append(v_f1)
        history["val_precision"].append(v_p)
        history["val_recall"].append(v_r)

        print(f"Epoch {epoch:2d} | loss={train_loss:.4f} | val_F1={v_f1:.4f} | P={v_p:.3f} | R={v_r:.3f}")

        if v_f1 > best_f1:
            best_f1 = v_f1
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    print(f"Best val F1: {best_f1:.4f}")
    return history

### Smoke test on 1000 samples

In [ ]:
small_loader = DataLoader(VulnDataset(X_train[:1000], y_train[:1000]),
                          batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
small_model = BiLSTMClassifier(len(vocab), emb_dim=64, hidden_dim=64).to(device)

n_pos = sum(y_train[:1000])
n_neg = 1000 - n_pos
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_neg/max(n_pos,1)*2.0], device=device))
optimizer = torch.optim.Adam(small_model.parameters(), lr=1e-3)

for epoch in range(1, 4):
    small_model.train()
    total = 0
    for x, lengths, y in small_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(small_model(x, lengths), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f"Smoke epoch {epoch}: loss={total/len(small_loader):.4f}")
print("Smoke test passed")

### Train 3 models

We train three models with different architectures and hyperparameters to create ensemble diversity. All use a boosted `pos_weight` (2× the natural class ratio) to improve recall on the minority vulnerable class.

| Model | Architecture | Embedding | Hidden | Layers | Dropout |
|-------|-------------|-----------|--------|--------|--------|
| 1 | BiLSTM | 128 | 128 | 1 | 0.3 |
| 2 | BiLSTM | 192 | 192 | 2 | 0.4 |
| 3 | BiGRU  | 160 | 160 | 2 | 0.35 |

In [ ]:
print("Training Model 1 — BiLSTM (128, 1 layer)")
torch.manual_seed(42)
model1 = BiLSTMClassifier(len(vocab), emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.3).to(device)
hist1 = train_model(model1, device)

In [ ]:
print("Training Model 2 — BiLSTM (192, 2 layers)")
torch.manual_seed(42)
model2 = BiLSTMClassifier(len(vocab), emb_dim=192, hidden_dim=192, num_layers=2, dropout=0.4).to(device)
hist2 = train_model(model2, device)

In [ ]:
print("Training Model 3 — BiGRU (160, 2 layers)")
torch.manual_seed(123)
model3 = BiGRUClassifier(len(vocab), emb_dim=160, hidden_dim=160, num_layers=2, dropout=0.35).to(device)
hist3 = train_model(model3, device)

### Loss and F1 graphs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for hist, name in [(hist1, "BiLSTM-128"), (hist2, "BiLSTM-192"), (hist3, "BiGRU-160")]:
    ep = range(1, len(hist["train_loss"]) + 1)
    axes[0].plot(ep, hist["train_loss"], 'o--', label=f"{name} train", markersize=4)
    axes[0].plot(ep, hist["val_loss"],   's-',  label=f"{name} val", markersize=4)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training and Validation Loss"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

for hist, name in [(hist1, "BiLSTM-128"), (hist2, "BiLSTM-192"), (hist3, "BiGRU-160")]:
    ep = range(1, len(hist["val_f1"]) + 1)
    axes[1].plot(ep, hist["val_f1"], 'o-', label=name, markersize=4)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("F1")
axes[1].set_title("Validation F1"); axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, "loss_curves.png"), dpi=150, bbox_inches='tight')
plt.show()

### Ensemble threshold tuning

With an imbalanced dataset, the default 0.5 threshold causes the model to under-predict the minority class. We sweep thresholds on the validation set to find the value that maximises F1, balancing precision and recall.

In [ ]:
val_probs = np.mean([get_probs(m, val_loader, device) for m in [model1, model2, model3]], axis=0)
val_ys = np.array(y_val)

print("Threshold |  Prec   Rec    F1")
print("-" * 40)
best_thresh, best_f1 = 0.5, 0.0
for t in np.arange(0.05, 0.55, 0.005):
    f = f1_score(val_ys, (val_probs > t).astype(int), zero_division=0)
    if f > best_f1:
        best_f1, best_thresh = f, t

for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    p = precision_score(val_ys, (val_probs > t).astype(int), zero_division=0)
    r = recall_score(val_ys, (val_probs > t).astype(int), zero_division=0)
    f = f1_score(val_ys, (val_probs > t).astype(int), zero_division=0)
    print(f"  {t:.2f}    | {p:.3f}  {r:.3f}  {f:.3f}")

THRESHOLD = 0.15
print(f"\nUsing threshold: {THRESHOLD}")

### Final evaluation on held-out test set

In [ ]:
test_probs = np.mean([get_probs(m, test_loader, device) for m in [model1, model2, model3]], axis=0)
test_preds = (test_probs > THRESHOLD).astype(int)
test_ys = np.array(y_test)

print(f"Accuracy : {accuracy_score(test_ys, test_preds):.4f}")
print(f"Precision: {precision_score(test_ys, test_preds, zero_division=0):.4f}")
print(f"Recall   : {recall_score(test_ys, test_preds, zero_division=0):.4f}")
print(f"F1 Score : {f1_score(test_ys, test_preds, zero_division=0):.4f}")
print()
print(classification_report(test_ys, test_preds, target_names=["safe", "vulnerable"]))

cm = confusion_matrix(test_ys, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Safe', 'Pred Vuln'],
            yticklabels=['True Safe', 'True Vuln'])
plt.title("Confusion Matrix — Ensemble (Test Set)")
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, "confusion_matrix.png"), dpi=150, bbox_inches='tight')
plt.show()

### Challenge predictions

In [ ]:
challenge_loader = DataLoader(ChallengeDataset(challenge_funcs),
                              batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=collate_fn, num_workers=2)

ch_probs = np.mean([get_probs(m, challenge_loader, device) for m in [model1, model2, model3]], axis=0)
ch_preds = (ch_probs > THRESHOLD).astype(int)

submission_df = pd.DataFrame({"vul_id": challenge_ids, "is_vul": ch_preds})
submission_path = os.path.join(DRIVE_DIR, "submission.csv")
submission_df.to_csv(submission_path, index=False)

print(f"Saved: {submission_path}")
print(f"Vulnerable: {ch_preds.sum()} | Safe: {(ch_preds == 0).sum()}")

In [ ]:
torch.save({
    "model1": model1.state_dict(),
    "model2": model2.state_dict(),
    "model3": model3.state_dict(),
    "threshold": THRESHOLD,
}, os.path.join(DRIVE_DIR, "part3_ensemble.pt"))

with open(os.path.join(DRIVE_DIR, "part3_vocab.pkl"), "wb") as f:
    pickle.dump(vocab, f)

print("Models and vocab saved to Drive.")